In [1]:
import os
import time

from pyspark.sql import SparkSession

os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Cluster Configuration ---
# Format: local-cluster[num_workers, cores_per_worker, memory_per_worker_in_MB]
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 6
MEMORY_PER_EXECUTOR_MB = 4096

MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"

print(f"Running in mode: {MASTER_URL}")

sp_s = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "6")
    .config("spark.executor.instances", NUM_EXECUTORS)
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.shuffle.partitions", "4")  # For tests, less than the default 200
    .getOrCreate()
)

sp_s.sparkContext.setLogLevel("WARN")

# --- 3. Configuration Check ---
print("Session created.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")

# Check the number of executors (may take a couple of seconds to start)
time.sleep(3)
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Active executors (checked via RDD): {num_executors}")

# --- 4. Distribution Test (Example) ---
# To make sure the task went to executors, not stayed on the driver
def print_executor_info(iterator):
    import os
    # Get the executor ID from the process environment variables
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

# Create a dataframe and apply a transformation
df = sp_s.range(0, 10, 1, 4)  # 4 partitions
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Where tasks were executed:")
for line in result:
    print(line)

sp_s

Running in mode: local-cluster[2, 6, 4096]


26/08/27 19:11:04 WARN Utils: Your hostname, MacBook-Pro-Danil.local resolves to a loopback address: 127.0.0.1; using 192.168.1.64 instead (on interface en0)
26/08/27 19:11:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/27 19:11:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/27 19:11:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Session created.
Driver Memory Config: 4g
Executor Memory Config: 4g


📊 Active executors (checked via RDD): 2

🖥️ Where tasks were executed:
Executor ID: Driver/Local, PID: 21606
Executor ID: Driver/Local, PID: 21605
Executor ID: Driver/Local, PID: 21612
Executor ID: Driver/Local, PID: 21613


# AA test tutorial 
AA test is important part of randomized controlled experiment, for example AB test. 

The objectives of the AA test are to verify the assumption of uniformity of samples as a result of the applied partitioning method, to select the best partition from the available ones, and to verify the applicability of statistical criteria for checking uniformity. 

For example, there is a hypothesis about the absence of dependence of features on each other. If this hypothesis is not followed, the AA test will fail.

[Wiki AA test](https://github.com/sb-ai-lab/HypEx/wiki/%D0%90%D0%90-Test) with more detailed description of terms for AA test.

<ul>
  <li><a href="#creation-of-a-new-test-dataset-with-synthetic-data">Creation of a new test dataset with synthetic data.
  <li><a href="#one-split-of-aa-test">One split of AA test.
  <li><a href="#aa-test">AA test.
  <li><a href="#aa-test-with-stratification">AA test with stratification.
</ul>

In [2]:
from hypex import AATest
from hypex.dataset import (
    ConstGroupRole,
    Dataset,
    InfoRole,
    StratificationRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils import BackendsEnum, create_test_data


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


## Creation of a new test dataset with synthetic data. 

In order to be able to work with our data in HypEx, first we need to convert it into `dataset`. It is important to mark the data fields by assigning the appropriate `roles`:
- TargetRole: a role for columns that contain features or predictor variables. Our split will be based on them. Applied by default if the role is not specified for the column.
- TreatmentRole: a role for columns that show the treatment or intervention.
- InfoRole: a role for columns that contain information about the data, such as user IDs. 

In [3]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    },
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,7.0,1.0,488.5,476.888889,33.0,M,E-commerce
1,1.0,8.0,1.0,527.0,476.666667,69.0,F,Logistics
2,2.0,4.0,1.0,465.0,504.333333,22.0,F,Logistics
3,3.0,7.0,1.0,469.0,481.222222,25.0,F,E-commerce
4,4.0,2.0,1.0,489.5,519.333333,27.0,M,E-commerce
...,...,...,...,...,...,...,...,...
9995,9995.0,11.0,1.0,478.0,434.333333,59.0,M,Logistics
9996,9996.0,1.0,1.0,536.0,515.111111,29.0,M,E-commerce
9997,9997.0,10.0,1.0,483.0,444.777778,24.0,F,Logistics
9998,9998.0,0.0,0.0,491.5,419.333333,26.0,M,E-commerce


In [4]:
data.roles

{'user_id': Info(<class 'int'>),
 'pre_spends': Target(<class 'float'>),
 'post_spends': Target(<class 'float'>),
 'gender': Stratification(<class 'str'>),
 'signup_month': Default(<class 'float'>),
 'treat': Default(<class 'float'>),
 'age': Default(<class 'float'>),
 'industry': Default(<class 'str'>)}

## AA test
Then we run the experiment on our prepared dataset, wrapped into ExperimentData. In this case we select one of the pre-assembled pipeline, AA_TEST.
We can set the number of iterations for simple execution. In this case the random states are the numbers of each iteration.

In [5]:
test = AATest(n_iterations=10)
result = test.execute(data)

100%|██████████| 10/10 [00:22<00:00,  2.20s/it]
[DEBUG _set_best_split] best_splitter_id = 'AASplitter┴rs 1┴'
[DEBUG _set_best_split] after execute: ds.columns = ['user_id', 'signup_month', 'treat', 'pre_spends', 'post_spends', 'age', 'gender', 'industry', 'AASplitter┴rs 1┴best']
[DEBUG _set_best_split] additional cols = ['AASplitter┴rs 1┴best']
[DEBUG AAPassedReporter] analyser_ids = {'AAScoreAnalyzer': {'analysis_tables': ['AAScoreAnalyzer┴┴aa score', 'AAScoreAnalyzer┴┴best split statistics']}}
[DEBUG AAPassedReporter] analyser_tables keys = ['aa score', 'best split statistics']
[DEBUG AAPassedReporter] best_split_stats.columns = Index(['splitter_id',
       'pre_spends┆stats GroupDifference mean┆pre_spends control',
       'pre_spends┆stats GroupDifference mean┆pre_spends test_1',
       'post_spends┆stats GroupDifference mean┆post_spends control',
       'post_spends┆stats GroupDifference mean┆post_spends test_1',
       'pre_spends GroupDifference control mean test_1',
       'pre

In [6]:
result.resume

,feature,group,TTest aa test,KSTest aa test,TTest best split,KSTest best split,result,control mean,test mean,difference,difference %
0,post_spends,test_1,OK,OK,OK,OK,OK,451.731028,451.723171,-0.007857,-0.001739
1,pre_spends,test_1,OK,OK,OK,OK,OK,487.110485,487.046540,-0.063945,-0.013127


**Interpretation of AA test results**

Each row in the table corresponds to a target feature being tested for equality between the control and test groups. Two statistical tests are used:

- **TTest**: tests if means are statistically different.
- **KSTest**: tests if distributions differ.

The `OK` / `NOT OK` labels show whether the difference is statistically significant. A `NOT OK` result indicates a possible imbalance.

Typical threshold:
- If p-value < 0.05 → `NOT OK` (statistically significant difference)
- If p-value ≥ 0.05 → `OK` (no significant difference)

If any metric has a `NOT OK` status in the `AA test` column, it means at least one iteration showed significant difference.


In [7]:
result.aa_score

,score,pass
pre_spends TTest test_1,0.95,True
post_spends TTest test_1,0.95,True
pre_spends KSTest test_1,0.95,True
post_spends KSTest test_1,0.95,True


**Interpreting `aa_score`**

This output shows p-values and the overall pass/fail status for each test type and feature. A high p-value (close to 1.0) means the test passed — the groups are similar.

- `score`: p-value of the statistical test.
- `pass`: True if no iterations showed significant differences.

Note: Even if the average p-value is high, the `pass` might still be False if at least one of the iterations had a p-value < 0.05.


In [8]:
result.best_split

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,split
2,2,4.0,1.0,465.0,504.333333,22.0,F,Logistics,control
4,4,2.0,1.0,489.5,519.333333,27.0,M,E-commerce,control
5,5,0.0,0.0,469.5,424.666667,30.0,F,Logistics,control
8,8,0.0,0.0,491.0,409.666667,18.0,M,E-commerce,test_1
12,12,0.0,0.0,490.5,422.888889,51.0,F,E-commerce,control
...,...,...,...,...,...,...,...,...,...
8996,9146,0.0,0.0,470.0,422.444444,40.0,M,E-commerce,test_1
8997,9156,0.0,0.0,491.5,423.666667,42.0,M,E-commerce,test_1
8998,9158,3.0,1.0,501.5,538.888889,61.0,F,Logistics,control
8999,9159,0.0,0.0,478.5,415.111111,68.0,M,Logistics,test_1


**About `best_split`**

This shows the best found split of the dataset, where control and test groups are as similar as possible in terms of target metrics.

You can use this split for future modeling or as a validation check before proceeding to actual experiments.


In [9]:
result.best_split_statistic

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,post_spends,test_1,451.731028,451.723171,-0.007857,-0.001739,OK,0.992444,OK,0.570539
1,pre_spends,test_1,487.110485,487.046540,-0.063945,-0.013127,OK,0.873188,OK,0.836922


**Understanding `best_split_statistic`**

This table contains detailed statistics for the best (most balanced) split found across all iterations. You can compare:

- Mean values in control vs test group.
- Absolute and relative differences.
- p-values for both tests.

Ideally, all rows should have `OK` in both TTest and KSTest columns, and small difference values (<1%).

In [10]:
result.experiments

,splitter_id,pre_spends GroupDifference control mean test_1,pre_spends GroupDifference test mean test_1,pre_spends GroupDifference difference test_1,pre_spends GroupDifference difference % test_1,post_spends GroupDifference control mean test_1,post_spends GroupDifference test mean test_1,post_spends GroupDifference difference test_1,post_spends GroupDifference difference % test_1,pre_spends TTest p-value test_1,...,post_spends TTest pass test_1,pre_spends KSTest p-value test_1,pre_spends KSTest pass test_1,post_spends KSTest p-value test_1,post_spends KSTest pass test_1,mean TTest p-value,mean TTest pass,mean KSTest p-value,mean KSTest pass,mean test score
0,AASplitter┴rs 0┴,486.868865,487.281860,0.412995,0.084827,451.585822,451.864639,0.278817,0.061742,0.302584,...,False,0.880049,False,0.850247,False,0.519700,0.0,0.865148,0.0,0.749999
1,AASplitter┴rs 1┴,487.110485,487.046540,-0.063945,-0.013127,451.731028,451.723171,-0.007857,-0.001739,0.873188,...,False,0.836922,False,0.570539,False,0.932816,0.0,0.703731,0.0,0.780092
2,AASplitter┴rs 2┴,487.092050,487.064028,-0.028021,-0.005753,452.248674,451.204832,-1.043842,-0.230811,0.944233,...,False,0.927353,False,0.192981,False,0.576248,0.0,0.560167,0.0,0.565527
3,AASplitter┴rs 3┴,486.990447,487.165667,0.175220,0.035980,451.600978,451.853136,0.252158,0.055837,0.661816,...,False,0.859989,False,0.812392,False,0.711485,0.0,0.836191,0.0,0.794622
4,AASplitter┴rs 4┴,487.092742,487.063588,-0.029154,-0.005985,451.484170,451.966008,0.481838,0.106723,0.941986,...,False,0.924072,False,0.649866,False,0.751674,0.0,0.786969,0.0,0.775204
5,AASplitter┴rs 5┴,486.989493,487.171266,0.181773,0.037326,450.969623,452.524363,1.554741,0.344755,0.650106,...,False,0.536780,False,0.165647,False,0.355536,0.0,0.351214,0.0,0.352655
6,AASplitter┴rs 6┴,486.876992,487.275077,0.398085,0.081763,451.596284,451.855184,0.258900,0.057330,0.320358,...,False,0.204357,False,0.894626,False,0.537670,0.0,0.549492,0.0,0.545551
7,AASplitter┴rs 7┴,487.148953,487.007534,-0.141419,-0.029030,451.876337,451.578575,-0.297761,-0.065894,0.724069,...,False,0.970789,False,0.255717,False,0.721855,0.0,0.613253,0.0,0.649454
8,AASplitter┴rs 8┴,487.279871,486.875278,-0.404593,-0.083031,451.205966,452.250557,1.044591,0.231511,0.312487,...,False,0.434609,False,0.385825,False,0.260212,0.0,0.410217,0.0,0.360215
9,AASplitter┴rs 9┴,487.208767,486.946365,-0.262402,-0.053858,451.789659,451.663966,-0.125693,-0.027821,0.512441,...,False,0.775830,False,0.977044,False,0.696005,0.0,0.876437,0.0,0.816293


# AA Test with random states

We can also adjust some of the preset parameters of the experiment by assigning them to the respective params of the experiment. I.e. here we set the range of the random states we want to run our AA test for. 

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    }, 
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

In [ ]:
test = AATest(random_states=[56, 72, 2, 43])
result = test.execute(data)

In [ ]:
result.resume

In [ ]:
result.aa_score

In [ ]:
result.best_split

In [ ]:
result.best_split_statistic

In [ ]:
result.experiments

# AA Test with stratification

Depending on your requirements it is possible to stratify the data. You can set `stratification=True` and `StratificationRole` in `Dataset` to run it with stratification.

Stratified AA tests ensure that both groups (control/test) have the same proportions of categories (e.g. same % of genders or regions). This prevents imbalances in categorical features that can distort results.

Make sure to assign `StratificationRole` to relevant columns in your dataset before enabling stratification.

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    }, 
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

In [ ]:
test = AATest(random_states=[56, 72, 2, 43], stratification=True)
result = test.execute(data)

In [ ]:
result.resume

In [ ]:
result.aa_score

In [ ]:
result.best_split

In [ ]:
result.best_split_statistic

In [ ]:
result.experiments

# AA Test by samples 

Depending on your requirements and size of data it is possible to estimate AA test on samples the data. You can set `sample_size=size` to run it. 

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    },
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

In [ ]:
test = AATest(n_iterations=10, sample_size=0.3)
result = test.execute(data)

In [ ]:
result.resume

In [ ]:
result.aa_score

In [ ]:
result.best_split

In [ ]:
result.best_split_statistic

In [ ]:
result.experiments

# AATest with Target Role for a categorical feature

It is possible to assign Target Role to categorical features. A categorical feature can also be the target or outcome variable. In this case, the Chi-square test is added to the pipeline of AATest.

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "treat": TreatmentRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": TargetRole(str)
    }, data=create_test_data(),
)
data

In [ ]:
test = AATest(n_iterations=10)
result = test.execute(data)

In [ ]:
result.resume

In [ ]:
result.aa_score

In [ ]:
result.best_split

In [ ]:
result.best_split_statistic

In [ ]:
result.experiments

# AATest with unequal group sizes

AATest can be performed to get a split with unequal the groups of different sizes by using `unequal_size` argument. Also Whelch correction can be applied by adding `t_test_equal_vat=False` argument while initiating AATest instance.

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    },
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

In [ ]:
test = AATest(n_iterations=10, control_size=0.3, t_test_equal_var=False)
result = test.execute(data)

In [ ]:
result.best_split.data.groupby("split").agg("count")

In [ ]:
result.best_split_statistic

# AAnTest

AAnTest is an extension of AATest that allows to split the dataset into several test groups, additionally to the control group.

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    },
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

In [ ]:
test = AATest(groups_sizes=[0.3, 0.2, 0.2, 0.3])
result = test.execute(data)

In [ ]:
result.best_split.data.groupby("split").agg("count")

In [ ]:
result.best_split_statistic

# AATest with partially pre-defined groups

Certain users can be pre-assigned to either the test or the control group, so that they are not randomly assigned. This can be done using the `ConstGroupRole` role. In order to pre-assign users to the control group they should have a value of `control`, and in the test group they should have a value of `test` in the column with the role `ConstGroupRole`. Users that are not pre-assigned to either the control or the test group should have `None`, so that they will be assigned randomly.

In [ ]:
pd_data= create_test_data()
pd_data.loc[pd_data["treat"]==0, "const_grp"] = "control"
pd_data.loc[pd_data["treat"]==1, "const_grp"] = "test"
pd_data.loc[2000:, "const_grp"] = None

data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "const_grp": ConstGroupRole(str),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
        "industry": TargetRole(str),
    }, data=pd_data,
    session=sp_s,
    backend=BackendsEnum.spark
)
data

In [ ]:
test = AATest(n_iterations=1)
result = test.execute(data)

In [ ]:
result.resume

In [ ]:
result.best_split

## Common issues and tips

- **Missing roles**: Make sure all target variables are assigned `TargetRole`. Columns without roles may cause silent failure.
- **Stratification**: If your dataset contains categorical features (e.g. `gender`, `region`) that may affect the outcome, use `StratificationRole` and enable `stratification=True` in `AATest(...)`.
- **Imbalanced categories**: If some categories have too few samples, stratified splits may become unstable. Consider filtering or merging rare categories.
- **Random fluctuations**: On small datasets, it's normal to see occasional `NOT OK` results. Use more iterations (e.g. `n_iterations=50`) for stability.
- **Missing values**: NaNs in stratification columns may be treated as separate categories. Clean or fill missing values before stratified AA tests.